In [ ]:
# Dependencias descritas no README.md
# Importacoes e configuracoes realizadas nas celulas seguintes


In [ ]:
from pathlib import Path
import pandas as pd
from pandasql import sqldf

# Centralizacao de caminhos para alta reprodutibilidade
DATA_DIR = Path("../dados")

pysqldf = lambda q: sqldf(q, globals())


In [ ]:
buyers = pd.read_csv(DATA_DIR / 'buyers.csv', sep=',')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv', sep=',')
orders = pd.read_csv(DATA_DIR / 'orders.csv', sep=',')
payments = pd.read_csv(DATA_DIR / 'payments.csv', sep=',')
products = pd.read_csv(DATA_DIR / 'products.csv', sep=',')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv', sep=',')


In [ ]:
# Desafio 1: Faturamento Bruto Mensal — Janela Movel dos Ultimos 12 Meses
# Decisoes Tecnicas e Premissas de Negocio:
# 1. Premissa Temporal (Janela Movel): Janela movel dos ultimos 12 meses em curso (01/12/2023 a 29/11/2024),
#    onde o mes atual (2024-11) reflete a volumetria acumulada ate a data maxima registrada na base.
# 2. Granularidade e Consistencia: Faturamento bruto recalculado na tabela 'order_items' (COALESCE(SUM(qty * unit_price), 0))
#    para manter alinhamento com a premissa de nao depender de 'orders.total_value'.
# 3. Engenharia Defensiva: Uso de COUNT(DISTINCT o.id) para contagem unica de pedidos.
# 4. Status Validos: Filtrados apenas 'completed' e 'delivered', descartando pedidos cancelados ou em processamento.

query = """
WITH order_gross AS (
    -- Recalcula o valor bruto por pedido a partir dos itens granulares
    SELECT 
        order_id,
        COALESCE(SUM(qty * unit_price), 0) AS valor_bruto_itens
    FROM order_items
    GROUP BY order_id
)
SELECT 
    strftime('%Y-%m', o.created_at) AS mes,
    ROUND(SUM(og.valor_bruto_itens), 2) AS faturamento_bruto,
    COUNT(DISTINCT o.id) AS quantidade_pedidos,
    ROUND(SUM(og.valor_bruto_itens) / COUNT(DISTINCT o.id), 2) AS ticket_medio
FROM orders o
JOIN order_gross og ON o.id = og.order_id
WHERE o.status IN ('completed', 'delivered')
  AND o.created_at >= date((SELECT MAX(created_at) FROM orders), 'start of month', '-11 months')
GROUP BY mes
ORDER BY mes DESC;
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


In [ ]:
# Desafio 2: Ranking dos 10 Sellers com Maior Crescimento de GMV (Trimestre Atual x Anterior)
# Decisoes Tecnicas e Premissas de Negocio:
# 1. Premissa de GMV: Na ausencia de uma definicao explicita no enunciado sobre a semantica de GMV e os status elegiveis, adotou-se 'orders.total_value' e consideraram-se estritamente pedidos efetivados ('completed', 'delivered') como premissa operacional para este exercicio.
# 2. Trimestre Parcial (Q4/2024): O trimestre atual (Q4/2024) contem dados ate 29/11/2024 (incompleto); a comparacao reflete o periodo disponivel em Q4/2024 contra o trimestre anterior completo (Q3/2024).
# 3. Analise do Resultado: A variacao negativa generalizada reflete a tendencia global de retracao da base no periodo; os 10 sellers ranqueados representam a menor taxa de retracao (maior crescimento relativo).

query = """
WITH seller_quarters AS (
    -- Agrupa GMV e pedidos por seller e trimestre apenas para vendas efetivadas
    SELECT 
        o.seller_id,
        s.name AS seller_name,
        s.state,
        strftime('%Y', o.created_at) || 'Q' || ((CAST(strftime('%m', o.created_at) AS INTEGER) + 2) / 3) AS yr_qtr,
        SUM(o.total_value) AS gmv,
        COUNT(o.id) AS qtd_pedidos
    FROM orders o
    JOIN sellers s ON o.seller_id = s.id
    WHERE o.status IN ('completed', 'delivered')
    GROUP BY 1, 2, 3, 4
),
ordered_quarters AS (
    -- Isola os 2 trimestres mais recentes com vendas na base
    SELECT DISTINCT yr_qtr
    FROM seller_quarters
    ORDER BY yr_qtr DESC
    LIMIT 2
),
pivot_quarters AS (
    -- Pivota o GMV e volume de pedidos para o trimestre anterior e atual
    SELECT 
        sq.seller_id,
        sq.seller_name,
        sq.state,
        MAX(CASE WHEN sq.yr_qtr = (SELECT MIN(yr_qtr) FROM ordered_quarters) THEN sq.gmv END) AS gmv_anterior,
        MAX(CASE WHEN sq.yr_qtr = (SELECT MAX(yr_qtr) FROM ordered_quarters) THEN sq.gmv END) AS gmv_atual,
        MAX(CASE WHEN sq.yr_qtr = (SELECT MIN(yr_qtr) FROM ordered_quarters) THEN sq.qtd_pedidos END) AS pedidos_anterior,
        MAX(CASE WHEN sq.yr_qtr = (SELECT MAX(yr_qtr) FROM ordered_quarters) THEN sq.qtd_pedidos END) AS pedidos_atual
    FROM seller_quarters sq
    WHERE sq.yr_qtr IN (SELECT yr_qtr FROM ordered_quarters)
    GROUP BY 1, 2, 3
)
SELECT 
    seller_name,
    state,
    ROUND(gmv_anterior, 2) AS gmv_anterior,
    ROUND(gmv_atual, 2) AS gmv_atual,
    ROUND(((gmv_atual - gmv_anterior) / gmv_anterior) * 100, 2) AS percentual_crescimento
FROM pivot_quarters
WHERE pedidos_anterior >= 50 AND pedidos_atual >= 50
ORDER BY percentual_crescimento DESC
LIMIT 10;
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2


In [ ]:
# Desafio 3: Identificacao de Pedidos com Desconto Abusivo (> 40% do Valor Bruto)
# Decisoes Tecnicas e Premissas de Negocio:
# 1. Aderencia Literal ao Enunciado: Exclusao estrita de pedidos cancelados ('status <> cancelled'), mantendo demais transacoes na analise.
# 2. Granularidade: Recalculo do valor bruto em 'order_items' (qty * unit_price) com COALESCE para evitar distorcoes de colunas agregadas ou valores nulos.
# 3. Protecao Defensiva: Filtro 'pv.valor_bruto > 0' para prevencao de erros de divisao por zero.

query = """
WITH pedido_valores AS (
    -- Recalcula o valor bruto real (quantidade * preco unitario) e soma dos descontos dos itens
    SELECT 
        order_id,
        COALESCE(SUM(qty * unit_price), 0) AS valor_bruto,
        COALESCE(SUM(discount), 0) AS desconto_total
    FROM order_items
    GROUP BY order_id
)
SELECT 
    o.id AS order_id,
    s.name AS seller_name,
    o.created_at AS data_pedido,
    ROUND(pv.valor_bruto, 2) AS valor_bruto,
    ROUND(pv.desconto_total, 2) AS desconto_total,
    ROUND((pv.desconto_total / pv.valor_bruto) * 100, 2) AS percentual_desconto
FROM orders o
JOIN pedido_valores pv ON o.id = pv.order_id
JOIN sellers s ON o.seller_id = s.id
-- Exclui apenas pedidos cancelados conforme requisito literal, previne divisao por zero e isola descontos > 40%
WHERE o.status <> 'cancelled'
  AND pv.valor_bruto > 0
  AND (pv.desconto_total / pv.valor_bruto) > 0.40;
"""

resultado_desafio3 = pysqldf(query)
display(resultado_desafio3.head(5))


In [ ]:
# Validação de Dados (Desafio 3): Estatísticas Consolidadas dos Pedidos Suspeitos
validacao_d3 = pysqldf("""
WITH pedido_valores AS (
    SELECT 
        order_id,
        COALESCE(SUM(qty * unit_price), 0) AS valor_bruto,
        COALESCE(SUM(discount), 0) AS desconto_total
    FROM order_items
    GROUP BY order_id
),
pedidos_suspeitos AS (
    SELECT 
        o.id AS order_id,
        s.name AS seller_name,
        pv.valor_bruto,
        pv.desconto_total,
        (pv.desconto_total / pv.valor_bruto) * 100 AS pct_desc
    FROM orders o
    JOIN pedido_valores pv ON o.id = pv.order_id
    JOIN sellers s ON o.seller_id = s.id
    WHERE o.status <> 'cancelled'
      AND pv.valor_bruto > 0
      AND (pv.desconto_total / pv.valor_bruto) > 0.40
)
SELECT 
    COUNT(*) AS total_pedidos_suspeitos,
    COUNT(DISTINCT seller_name) AS total_sellers_envolvidos,
    ROUND(AVG(pct_desc), 2) AS desconto_medio_pct,
    ROUND(MAX(pct_desc), 2) AS desconto_maximo_pct
FROM pedidos_suspeitos;
""")
display(validacao_d3)


In [ ]:
# Desafio 4: Identificacao de Anomalia em Produtos (Alto Volume Vendido sem nunca ser o Item de Maior Valor Unitario)
# Decisoes Tecnicas, Diagnostico e Prova Quantitativa:
# 1. Implementacao Estrita: Ranqueamento dos itens por pedido via RANK() OVER(PARTITION BY order_id ORDER BY unit_price DESC).
# 2. Diagnostico Demonstrado: 100% dos produtos que aparecem na base atingiram Rank 1 em pelo menos um pedido; portanto, a condicao MIN(rank_valor_unitario) > 1 nao retorna registros.
# 3. A ocorrencia de pedidos de um unico item contribui para o comportamento observado, pois nesses pedidos o unico item necessariamente ocupa o Rank 1.

query = """
WITH RankedItems AS (
    -- Ranqueia os itens de cada pedido pelo preco unitario (decrescente)
    SELECT 
        order_id,
        product_id,
        qty,
        unit_price,
        RANK() OVER(PARTITION BY order_id ORDER BY unit_price DESC) as rank_valor_unitario
    FROM order_items
),
ProductStats AS (
    -- Consolida total vendido e a melhor posicao obtida pelo produto em todos os pedidos
    SELECT 
        product_id,
        SUM(qty) AS total_unidades_vendidas,
        MIN(rank_valor_unitario) AS melhor_posicao_valor_unitario
    FROM RankedItems
    GROUP BY product_id
)
SELECT 
    ps.product_id,
    p.name AS product_name,
    ps.total_unidades_vendidas
FROM ProductStats ps
JOIN products p ON ps.product_id = p.id
-- Filtra produtos com volume > 1000 unidades que NUNCA atingiram a 1ª posicao (> 1)
WHERE ps.total_unidades_vendidas > 1000
  AND ps.melhor_posicao_valor_unitario > 1;
"""

resultado_desafio4 = pysqldf(query)
display(resultado_desafio4)


In [ ]:
# Validação de Dados (Desafio 4): Prova Quantitativa do Diagnóstico de 0 Linhas
validacao_d4 = pysqldf("""
WITH OrderItemCounts AS (
    SELECT 
        order_id,
        COUNT(*) AS qtd_itens
    FROM order_items
    GROUP BY order_id
),
RankedItems AS (
    SELECT 
        oi.order_id,
        oi.product_id,
        ic.qtd_itens,
        RANK() OVER(PARTITION BY oi.order_id ORDER BY oi.unit_price DESC) as rank_valor_unitario
    FROM order_items oi
    JOIN OrderItemCounts ic ON oi.order_id = ic.order_id
),
ProductStats AS (
    SELECT 
        product_id,
        MIN(rank_valor_unitario) AS melhor_posicao_geral,
        SUM(CASE WHEN rank_valor_unitario = 1 THEN 1 ELSE 0 END) AS vezes_rank_1_geral,
        SUM(CASE WHEN rank_valor_unitario = 1 AND qtd_itens = 1 THEN 1 ELSE 0 END) AS vezes_rank_1_pedido_unico,
        SUM(CASE WHEN rank_valor_unitario = 1 AND qtd_itens > 1 THEN 1 ELSE 0 END) AS vezes_rank_1_pedido_composto
    FROM RankedItems
    GROUP BY product_id
)
SELECT 
    COUNT(*) AS total_produtos_base,
    SUM(CASE WHEN melhor_posicao_geral = 1 THEN 1 ELSE 0 END) AS prods_com_rank_1,
    SUM(CASE WHEN melhor_posicao_geral > 1 THEN 1 ELSE 0 END) AS prods_sempre_abaixo_rank_1,
    SUM(CASE WHEN vezes_rank_1_pedido_unico > 0 THEN 1 ELSE 0 END) AS prods_rank_1_em_pedido_unico
FROM ProductStats;
""")
display(validacao_d4)
